# Matrix Event Dashboard Example

Listens to ML events (metrics, status, model info) from a Matrix room and prints/logs them in real time.
- Requires: `matrix-nio[http]` and project matrix wrapper lib.
- Use environment variables/config for credentials in production!
- Ready for integration with plotting libraries.


In [ ]:
import os
import asyncio
from libs.api-clients.matrix_wrapper import MatrixClientWrapper
from nio import RoomMessageText

# Credentials: override as needed, or use os.environ.get('...')
MATRIX_HOMESERVER = os.environ.get('MATRIX_HOMESERVER', 'https://matrix.org')
MATRIX_BOT_USER = os.environ.get('MATRIX_BOT_USER', '@botuser:matrix.org')
MATRIX_BOT_PASSWORD = os.environ.get('MATRIX_BOT_PASSWORD', 'yourpassword')
MATRIX_ROOM_ID = os.environ.get('MATRIX_ROOM_ID', '!yourroomid:matrix.org')

print('Connecting with credentials:', MATRIX_BOT_USER, MATRIX_ROOM_ID)


In [ ]:
class MatrixDashboard(MatrixClientWrapper):
    async def listen_events(self, event_types=None):
        await self.login()
        print('Listening for events... (Ctrl+C to stop)')
        self.client.add_event_callback(self._event_callback, RoomMessageText)
        await self.client.join(self.room_id)
        while True:
            await self.client.sync(timeout=30000)
    
    async def _event_callback(self, room, event):
        try:
            event_type = getattr(event, 'msgtype', 'unknown')
            content = event.body if hasattr(event, 'body') else event.source.get('metrics')
            print(f'Event from {room.display_name}: Type={event_type}, Content={content}')
        except Exception as e:
            print('Error parsing event:', e)


In [ ]:
# Async listener runner
async def run_dashboard():
    dashboard = MatrixDashboard(MATRIX_HOMESERVER, MATRIX_BOT_USER, MATRIX_BOT_PASSWORD, MATRIX_ROOM_ID)
    await dashboard.listen_events(event_types=['ml.metrics', 'ml.status', 'ml.model'])

try:
    asyncio.run(run_dashboard())
except KeyboardInterrupt:
    print('Stopped by user.')


---
**Usage Notes:**
- Install Jupyter requirements and matrix-nio for this notebook.
- Use environment variables for Matrix bot config (homeserver, user, password, room) for security.
- Handles standard event types: `ml.metrics`, `ml.status`, `ml.model` (easy to extend).
- Ready to add Matplotlib/Plotly code for visual dashboards.
- For Docker use, mount the notebook folder and set environment variables as needed.
